# Hours vs Score — Linear Regression (All Phases)

**Goal**: Predict a student's exam score from hours studied.

| Phase | Description |
|---|---|
| 1 | Data Loading & Exploration |
| 2 | Data Cleaning |
| 3 | Feature Check |
| 4 | Train / Test Split |
| 5 | Model Training |
| 6 | Model Evaluation |
| 7 | Visualization |
| 8 | Predict New Value |

## Phase 1 — Data Loading & Exploration (EDA)

Always explore data before modelling:
- How many rows/columns?
- Any missing values?
- What is the range and distribution of values?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Load the CSV ---
df = pd.read_csv('hours_score.csv')

print('=== Dataset Shape ===')
print(f'{df.shape[0]} rows x {df.shape[1]} columns')

print('\n=== Full Data ===')
print(df.to_string(index=False))  # show all rows (small dataset)

print('\n=== Data Types ===')
print(df.dtypes)

# isnull().sum() counts NaN per column
print('\n=== Missing Values ===')
print(df.isnull().sum())

# describe() gives count, mean, std, min, 25%, 50%, 75%, max
print('\n=== Statistics ===')
print(df.describe().round(2))

# Correlation: how strongly Hours and Score move together
# 1.0 = perfect positive, -1.0 = perfect negative, 0 = no relationship
print('\n=== Correlation Matrix ===')
print(df.corr(numeric_only=True).round(3))


## Phase 2 — Data Cleaning

Check for:
- Missing values (drop or impute)
- Duplicate rows
- Outliers (values far outside normal range)

In [ ]:
# Check duplicates
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')
if duplicates > 0:
    df = df.drop_duplicates()  # remove duplicate rows
    print(f'Removed {duplicates} duplicate(s)')

# Check for outliers using IQR method
# IQR = Q3 - Q1; anything beyond Q1-1.5*IQR or Q3+1.5*IQR is an outlier
for col in df.columns:
    Q1  = df[col].quantile(0.25)  # 25th percentile
    Q3  = df[col].quantile(0.75)  # 75th percentile
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: range [{df[col].min()}, {df[col].max()}]  |  outlier bounds [{lower:.1f}, {upper:.1f}]  |  outliers found: {len(outliers)}')

print('\nData is clean — no issues found.')
print(f'Final dataset: {df.shape[0]} rows')


## Phase 3 — Feature Check

For this dataset:
- `Hours` = input feature (X) — already numeric, no encoding needed
- `Score` = target variable (y) — what we want to predict

> Both columns are numeric — no label encoding or one-hot encoding required.

In [ ]:
# X must be 2D — double brackets [[]] give shape (n_samples, 1)
# Single bracket [] gives shape (n_samples,) which sklearn rejects
X = df[['Hours']]  # 2D input — required by sklearn
y = df['Score']    # 1D target — just the values

print(f'X shape: {X.shape}  (rows=samples, cols=features)')
print(f'y shape: {y.shape}  (one score per sample)')

print('\nX (features):')
print(X.to_string())
print('\ny (target):')
print(y.to_string())


## Phase 4 — Train / Test Split

**This is the most important step to avoid misleading results.**

Split data into two sets:
- **Train set** — model learns patterns from this
- **Test set** — held out, never seen during training — used only for evaluation

> If you evaluate on training data, the model looks better than it really is.
> You must test on data the model has **never seen**.

**Note**: With only 6 rows, any split is very small. In practice always use 50+ rows.

In [ ]:
# test_size=0.2 = 20% of data goes to test set, 80% to train
# random_state=42 = fixed seed so split is identical every run
# shuffle=True = randomly mix rows before splitting (default True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

print('=== Split Result ===')
print(f'Total samples  : {len(X)}')
print(f'Training set   : {len(X_train)} rows ({len(X_train)/len(X)*100:.0f}%) — model learns from this')
print(f'Test set       : {len(X_test)} rows  ({len(X_test)/len(X)*100:.0f}%) — evaluation only, never used in training')

print('\nTraining data (X_train, y_train):')
train_df = X_train.copy()
train_df['Score'] = y_train
print(train_df.to_string())

print('\nTest data (X_test, y_test):')
test_df = X_test.copy()
test_df['Score'] = y_test
print(test_df.to_string())

# --- Feature Scaling (optional for Linear Regression, good habit) ---
# StandardScaler: makes mean=0, std=1 for each feature
scaler = StandardScaler()

# RULE: fit_transform on TRAIN only — scaler learns from training data
X_train_scaled = scaler.fit_transform(X_train)

# RULE: transform only on TEST — apply same stats, no re-fitting
# Re-fitting on test would let test data influence the scaler (data leakage)
X_test_scaled  = scaler.transform(X_test)

print(f'\nScaler mean (learned from train): {scaler.mean_[0]:.2f} hours')
print(f'Scaler std  (learned from train): {scaler.scale_[0]:.2f} hours')


## Phase 5 — Model Training

Linear Regression finds the best straight line through training data:
```
Score = slope × Hours + intercept
```
- **slope (coef_)** — how many marks increase per extra hour of study
- **intercept_** — predicted score when hours = 0 (base score)

In [ ]:
model = LinearRegression()  # create model object (no learning yet)

# fit() does the actual learning — finds best slope and intercept
# Only uses X_train and y_train — test set is never touched here
model.fit(X_train_scaled, y_train)

print('=== What the Model Learned ===')
print(f'Intercept (base score) : {model.intercept_:.2f}')
print(f'Slope (coef_)          : {model.coef_[0]:.2f}')
print(f'\nFormula: Score = {model.coef_[0]:.2f} x (scaled Hours) + {model.intercept_:.2f}')
print(f'\nInterpretation: each 1-unit increase in scaled hours adds ~{model.coef_[0]:.1f} marks')


## Phase 6 — Model Evaluation

**Always evaluate on the test set — not the training set.**

| Metric | What it means | Ideal |
|---|---|---|
| **MAE** | Average error in marks | Low |
| **MSE** | Mean squared error — penalises big mistakes | Low |
| **RMSE** | Root MSE — same unit as score | Low |
| **R²** | % of variance explained by the model | Close to 1.0 |

In [ ]:
# Predict on TRAIN to check training fit
y_pred_train = model.predict(X_train_scaled)

# Predict on TEST — this is the honest evaluation
y_pred_test  = model.predict(X_test_scaled)

def show_metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    print(f'\n--- {label} ---')
    print(f'  MAE  : {mae:.2f}   -> predictions off by ~{mae:.1f} marks on average')
    print(f'  MSE  : {mse:.2f}  -> mean squared error (bigger errors hurt more)')
    print(f'  RMSE : {rmse:.2f}   -> typical error is ~{rmse:.1f} marks')
    print(f'  R2   : {r2:.4f}  -> model explains {r2*100:.1f}% of the variance')
    return r2

r2_train = show_metrics(y_train, y_pred_train, 'Training Set')
r2_test  = show_metrics(y_test,  y_pred_test,  'Test Set (unseen data)')

# Overfit check: if train R2 >> test R2, model memorised training data
gap = r2_train - r2_test
print(f'\n=== Overfit Check ===')
print(f'  Train R2 : {r2_train:.4f}')
print(f'  Test  R2 : {r2_test:.4f}')
print(f'  Gap      : {gap:.4f}  ->', 'Good fit' if gap < 0.05 else 'Overfit risk' if gap < 0.15 else 'Overfitting!')

# Actual vs Predicted table
print('\n=== Test Set: Actual vs Predicted ===')
result_df = X_test.copy()
result_df['Actual Score']    = y_test.values
result_df['Predicted Score'] = y_pred_test.round(1)
result_df['Error']           = (y_test.values - y_pred_test).round(1)
print(result_df.to_string(index=False))


## Phase 7 — Visualization

3 key plots:
1. **Scatter + Regression Line** — does the line fit the data?
2. **Actual vs Predicted** — points near the diagonal = accurate model
3. **Residual Plot** — errors should be random around 0

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Hours vs Score — Linear Regression Analysis', fontsize=13, fontweight='bold')

# --- Plot 1: Data + Regression Line ---
ax1 = axes[0]
# Plot all data points
ax1.scatter(X_train, y_train, color='steelblue', s=80, label='Train data', zorder=3)
ax1.scatter(X_test,  y_test,  color='orange',    s=80, label='Test data',  zorder=3, marker='D')

# Draw the regression line across the full hour range
hour_range = np.linspace(X['Hours'].min(), X['Hours'].max(), 100).reshape(-1, 1)
hour_range_scaled = scaler.transform(hour_range)  # must scale new input same way
line_preds = model.predict(hour_range_scaled)
ax1.plot(hour_range, line_preds, color='red', lw=2, label='Regression line')

ax1.set_title('Data + Regression Line')
ax1.set_xlabel('Study Hours'); ax1.set_ylabel('Score')
ax1.legend(); ax1.grid(True, alpha=0.3)

# --- Plot 2: Actual vs Predicted (Test Set) ---
# Perfect model: all points lie exactly on the diagonal
ax2 = axes[1]
ax2.scatter(y_test, y_pred_test, color='mediumseagreen', s=100, zorder=3)
lo, hi = y.min() - 2, y.max() + 2
ax2.plot([lo, hi], [lo, hi], 'r--', lw=2, label='Perfect prediction')  # diagonal line
ax2.set_title(f'Actual vs Predicted  (R2={r2_test:.3f})')
ax2.set_xlabel('Actual Score'); ax2.set_ylabel('Predicted Score')
ax2.legend(); ax2.grid(True, alpha=0.3)

# --- Plot 3: Residual Plot ---
# residual = actual - predicted
# Good model: residuals scattered randomly around 0
# Pattern in residuals = model is missing something
residuals_train = y_train.values - y_pred_train
ax3 = axes[2]
ax3.scatter(y_pred_train, residuals_train, color='steelblue', s=80, label='Train', zorder=3)
ax3.scatter(y_pred_test,  y_test.values - y_pred_test, color='orange', s=80, marker='D', label='Test', zorder=3)
ax3.axhline(0, color='red', linestyle='--', lw=2)  # zero-error reference line
ax3.set_title('Residual Plot\n(errors should scatter randomly around 0)')
ax3.set_xlabel('Predicted Score'); ax3.set_ylabel('Residual (Actual - Predicted)')
ax3.legend(); ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Phase 8 — Predict New Value

Use the trained model to predict a score for a new student.
- Must **scale** the new input using the **same scaler** from training
- Never re-fit the scaler on new data

In [ ]:
# Test multiple hour values instead of using input() which blocks notebooks
test_hours = [1.5, 3, 4.5, 7, 10]

print('=== Predictions for New Students ===')
print(f'{"Hours Studied":<16} {"Predicted Score":<18} {"Confidence Range"}')
print('-' * 55)

rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

for h in test_hours:
    # Step 1: create 2D array — required by sklearn
    new_input = np.array([[h]])

    # Step 2: scale using the SAME scaler fitted on training data
    new_scaled = scaler.transform(new_input)

    # Step 3: predict — returns array, [0][0] gets the single value
    pred = model.predict(new_scaled)[0]
    pred = np.clip(pred, 0, 100)  # clamp score to valid range 0-100

    low  = np.clip(pred - rmse_test, 0, 100)
    high = np.clip(pred + rmse_test, 0, 100)
    print(f'{h:<16} {pred:<18.1f} {low:.1f} – {high:.1f}')

print(f'\nModel RMSE: {rmse_test:.2f} marks  (typical error on test set)')
print(f'Confidence range = predicted score +/- {rmse_test:.2f} marks')


## Summary — All 8 Phases

```
Phase 1  EDA         shape, dtypes, describe(), correlation
Phase 2  Cleaning    duplicates, missing values, outlier check (IQR)
Phase 3  Features    X = Hours (2D), y = Score (1D)
Phase 4  Split       train_test_split(X, y, test_size=0.2, random_state=42)
                     scaler.fit_transform(X_train) — fit on train only
                     scaler.transform(X_test)      — apply same stats to test
Phase 5  Training    model.fit(X_train_scaled, y_train)
                     inspect coef_ (slope) and intercept_
Phase 6  Evaluation  MAE / MSE / RMSE / R2 on TEST set only
                     overfit check: train R2 vs test R2 gap
Phase 7  Plots       data+line, actual vs predicted, residuals
Phase 8  Predict     scaler.transform(new_input) -> model.predict()
```

**Phase 4 key rules (Train/Test Split)**
- Always split BEFORE scaling or training
- `fit_transform` on train — scaler learns mean/std from training data only
- `transform` only on test — same stats applied, no new learning
- Never evaluate on training data — always use the test set
- `random_state=42` keeps the split identical every run